In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Load the cleaned dataset
data = pd.read_csv('../datasets/cleaned.csv')
print(f"Dataset shape: {data.shape}")

# Target column
target_col = 'Overall'

# Keep numeric features only to avoid encoding categoricals for now
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
feature_cols = [col for col in numeric_cols if col != target_col]

# Drop rows where the target is missing
data = data.dropna(subset=[target_col])
X = data[feature_cols]
y = data[target_col]

# Fill missing numeric values with column medians
X = X.fillna(X.median())

print(f"Number of numeric features: {len(feature_cols)}")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build pipeline with scaling and KNN regressor
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

knn_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=1
)

knn_search.fit(X_train, y_train)

print('Best params:', knn_search.best_params_)
print('Best CV RMSE:', (-knn_search.best_score_) ** 0.5)

In [ ]:
best_model = knn_search.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"Test MAE: {mae:.3f}")
print(f"Test RMSE: {rmse:.3f}")
print(f"Test R^2: {r2:.3f}")